# Handling Big Data with Dask ⚡

### ISA 383: Python for Business Analytics

Every dataset we have used so far fits comfortably in memory: the *Titanic* file has 891 rows.
Business data is often far larger: a retailer's point-of-sale log, a website's clickstream, or a telecom's call records can run to hundreds of millions of rows spread over many files.
This notebook introduces three ideas that let Python work with data that is too big or too slow for plain pandas: **chunking**, **lazy evaluation**, and **distributed (parallel) computing**, using the Dask library.

## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.

## Learning objectives 🎯

By the end of this notebook, you should be able to:

1. Explain why pandas runs into memory and speed limits, and measure a DataFrame's memory use.
2. Reduce memory with better data types and the columnar Parquet format.
3. Process data that does not fit in memory by reading it in chunks, and identify which calculations can be combined across chunks.
4. Explain lazy evaluation and task graphs, and use Dask DataFrames with `.compute()`, `dask.compute()`, and `.persist()`.
5. Describe the basics of distributed computing (partitions, tasks, schedulers, workers) and run a computation on a local Dask cluster.

## 1. Why big data is different 🐘

pandas is **eager** and **in-memory**: every command runs immediately, and the whole table must sit in your computer's RAM.
pandas often needs more RAM than the size of the file on disk, sometimes several times more, because text must be parsed into in-memory structures and many operations create temporary copies.

| Dataset | Rows | Typical size | Plain pandas on a laptop? |
|---|---|---|---|
| *Titanic* passenger list | 891 | 60 KB | instant |
| This notebook's transaction log | 1.2 million | about 90 MB as CSV | fine, but you can feel it |
| One year of point-of-sale data for a national retailer | hundreds of millions | tens of GB | runs out of memory |
| A large website's clickstream | billions | terabytes | needs a cluster |

There are three escape routes, and this notebook takes them in order:

1. **Shrink the data** with better types and formats, so that more of it fits.
2. **Chunking**: read a manageable piece, summarize it, throw it away, and move on.
3. **Lazy, parallel computing with Dask**: describe the whole computation first, then let Dask split it into many small tasks and run them on all your processor cores, or on many machines.

**Scenario.**
You are an analyst at an electronics retailer with stores in all seven emirates.
Each month, IT exports the month's transactions as a separate CSV file.
The data is **synthetic**, generated below with a fixed random seed so that everyone gets the same files.

## 2. Setup and data generation 🧰

The first cell makes sure Dask and its `distributed` scheduler (used in Section 6) are installed.
The generated data files are written to a temporary folder under `/content`.

In [ ]:
%pip install -q "dask[dataframe,distributed]"

In [2]:
import time
from pathlib import Path

import dask
import dask.dataframe as dd
import numpy as np
import pandas as pd
from IPython.display import display

print("pandas", pd.__version__, "| dask", dask.__version__)

pandas 2.2.3 | dask 2025.1.0


In [3]:
DATA_DIR = Path("/content/isa383/17_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

EMIRATES = ["Dubai", "Abu Dhabi", "Sharjah", "Ajman", "Ras Al Khaimah", "Fujairah", "Umm Al Quwain"]
PRODUCTS = {  # product: (category, lowest price, highest price) in AED
    "Laptop": ("Computing", 2500, 8000),
    "Monitor": ("Computing", 500, 2500),
    "Keyboard": ("Computing", 50, 400),
    "Mouse": ("Computing", 30, 300),
    "Phone": ("Mobile", 1500, 5500),
    "Tablet": ("Mobile", 1000, 4000),
    "Headphones": ("Accessories", 100, 1500),
    "Charger": ("Accessories", 40, 250),
}
ROWS_PER_MONTH = 100_000
rng = np.random.default_rng(383)


def make_month(month):
    # Generate one month of synthetic transactions.
    days = pd.date_range(f"2025-{month:02d}-01", periods=28, freq="D").strftime("%Y-%m-%d")
    product = rng.choice(list(PRODUCTS), ROWS_PER_MONTH)
    low = np.array([PRODUCTS[p][1] for p in product])
    high = np.array([PRODUCTS[p][2] for p in product])
    df = pd.DataFrame(
        {
            "transaction_id": np.arange(ROWS_PER_MONTH) + month * 1_000_000,
            "date": rng.choice(days, ROWS_PER_MONTH),
            "customer_id": rng.integers(1, 80_001, ROWS_PER_MONTH),
            "store_emirate": rng.choice(EMIRATES, ROWS_PER_MONTH, p=[0.35, 0.25, 0.15, 0.08, 0.08, 0.05, 0.04]),
            "product": product,
            "category": [PRODUCTS[p][0] for p in product],
            "quantity": rng.integers(1, 6, ROWS_PER_MONTH),
            "unit_price": (low + (high - low) * rng.random(ROWS_PER_MONTH)).round(2),
            "discount_pct": rng.choice([0, 5, 10, 15, 20, np.nan], ROWS_PER_MONTH),
            "payment_method": rng.choice(["Card", "Cash", "Digital wallet", "Instalments"], ROWS_PER_MONTH),
        }
    )
    df["total"] = (df["quantity"] * df["unit_price"]).round(2)
    return df


start = time.perf_counter()
for month in range(1, 13):
    make_month(month).to_csv(DATA_DIR / f"transactions_2025-{month:02d}.csv", index=False)

files = sorted(DATA_DIR.glob("transactions_2025-*.csv"))
total_mb = sum(f.stat().st_size for f in files) / 1024**2
print(f"{len(files)} files, {total_mb:.1f} MB on disk, created in {time.perf_counter() - start:.1f} s")

12 files, 93.0 MB on disk, created in 2.7 s


In [4]:
pd.read_csv(files[0], nrows=5)

,transaction_id,date,customer_id,store_emirate,product,category,quantity,unit_price,discount_pct,payment_method,total
0,1000000,2025-01-14,4581,Abu Dhabi,Keyboard,Computing,5,284.71,5.0,Card,1423.55
1,1000001,2025-01-12,35701,Fujairah,Laptop,Computing,2,4438.68,10.0,Cash,8877.36
2,1000002,2025-01-11,50706,Ras Al Khaimah,Headphones,Accessories,4,1332.78,0.0,Card,5331.12
3,1000003,2025-01-12,63188,Dubai,Headphones,Accessories,2,727.71,0.0,Card,1455.42
4,1000004,2025-01-03,74030,Dubai,Monitor,Computing,3,2389.01,0.0,Instalments,7167.03


## 3. pandas first: measure, then shrink 🧮

Before reaching for a new tool, measure what you have.
`memory_usage(deep=True)` reports the real memory a DataFrame uses, including the text inside string columns.
Two cheap fixes often cut memory dramatically: **better types** (`category` for repeated labels, smaller integer types) and a **better file format** (Parquet).

### 🧭 Syntax first: Measuring and reducing memory

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.memory_usage(deep=True).sum() / 1024**2          # megabytes
pd.read_csv(path, usecols=[...], dtype={"col": "category"})
pd.to_numeric(df["col"], downcast="integer")
df.to_parquet(path, index=False)
pd.read_parquet(path, columns=["col_a", "col_b"])   # read only what you need
```

Documentation: [memory_usage](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.memory_usage.html) · [read_parquet](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html) · [Scaling to large datasets](https://pandas.pydata.org/docs/user_guide/scale.html)

In [5]:
def megabytes(df):
    return df.memory_usage(deep=True).sum() / 1024**2


start = time.perf_counter()
year = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
read_seconds = time.perf_counter() - start

print(f"Rows: {len(year):,}")
print(f"Read time: {read_seconds:.1f} s | memory: {megabytes(year):.0f} MB (files on disk: {total_mb:.0f} MB)")
year.dtypes

Rows: 1,200,000


Read time: 0.5 s | memory: 383 MB (files on disk: 93 MB)


transaction_id      int64
date               object
customer_id         int64
store_emirate      object
product            object
category           object
quantity            int64
unit_price        float64
discount_pct      float64
payment_method     object
total             float64
dtype: object

In [6]:
# Repeated labels -> category; small whole numbers -> smaller integer types.
compact = year.copy()
for column in ["store_emirate", "product", "category", "payment_method"]:
    compact[column] = compact[column].astype("category")
for column in ["quantity", "customer_id"]:
    compact[column] = pd.to_numeric(compact[column], downcast="integer")
compact["date"] = pd.to_datetime(compact["date"])

print(f"Before: {megabytes(year):.0f} MB -> after: {megabytes(compact):.0f} MB "
      f"({1 - megabytes(compact) / megabytes(year):.0%} smaller)")

Before: 383 MB -> after: 56 MB (85% smaller)


In [7]:
# Parquet stores data by column, compressed, with its types. Write once, then compare.
parquet_path = DATA_DIR / "transactions_2025.parquet"
compact.to_parquet(parquet_path, index=False)

start = time.perf_counter()
pd.read_parquet(parquet_path, columns=["store_emirate", "total"])
parquet_seconds = time.perf_counter() - start

print(f"Parquet file: {parquet_path.stat().st_size / 1024**2:.0f} MB vs {total_mb:.0f} MB of CSV")
print(f"Reading two columns: {parquet_seconds:.2f} s from Parquet vs {read_seconds:.1f} s for the full CSV read")

Parquet file: 25 MB vs 93 MB of CSV
Reading two columns: 0.01 s from Parquet vs 0.5 s for the full CSV read


These fixes are worth knowing because they are free: same library, same code.
But they only delay the problem.
When the data is ten or a hundred times larger than your RAM, no data type will save you, and you need to stop loading everything at once.

> 💭 **Quick thought question:** Why does converting `store_emirate` to `category` save so much memory, while converting `transaction_id` would not?

## 4. Chunking: one piece at a time 🧩

Chunking means reading a file in pieces of, say, 200,000 rows, summarizing each piece, keeping only the small summary, and then combining the summaries at the end.
Only one chunk is ever in memory, so the file can be far larger than your RAM.

```text
big file ──┬─> chunk 1 ──> partial sums ──┐
           ├─> chunk 2 ──> partial sums ──┼──> combine ──> final answer
           └─> chunk n ──> partial sums ──┘
```

The catch: the calculation must **decompose**, meaning the answer for the whole file can be built from answers for the pieces.

### 🧭 Syntax first: Reading in chunks

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
reader = pd.read_csv(path, chunksize=200_000)   # an iterator, not a DataFrame
partial_results = []
for chunk in reader:                            # each chunk is a normal DataFrame
    partial_results.append(chunk.groupby("key")["value"].sum())
final = pd.concat(partial_results).groupby(level=0).sum()
```

Documentation: [Iterating through files chunk by chunk](https://pandas.pydata.org/docs/user_guide/io.html#iterating-through-files-chunk-by-chunk)

In [8]:
reader = pd.read_csv(files[0], chunksize=40_000)
print(type(reader))

for number, chunk in enumerate(reader, start=1):
    print(f"chunk {number}: {len(chunk):>6,} rows, {megabytes(chunk):5.1f} MB")

<class 'pandas.io.parsers.readers.TextFileReader'>
chunk 1: 40,000 rows,  12.7 MB


chunk 2: 40,000 rows,  12.8 MB
chunk 3: 20,000 rows,   6.4 MB


In [9]:
# Revenue by emirate for the whole year, never holding more than one chunk in memory.
partial_sums = []
start = time.perf_counter()
for path in files:
    for chunk in pd.read_csv(path, usecols=["store_emirate", "total"], chunksize=200_000):
        partial_sums.append(chunk.groupby("store_emirate")["total"].sum())

revenue_chunked = pd.concat(partial_sums).groupby(level=0).sum().sort_values(ascending=False)
print(f"Chunked computation: {time.perf_counter() - start:.1f} s, {len(partial_sums)} partial results")

# Check against the full in-memory answer.
revenue_full = year.groupby("store_emirate")["total"].sum()
print("Matches the full pandas answer:", np.allclose(revenue_chunked.sort_index(), revenue_full.sort_index()))
(revenue_chunked / 1e6).round(1).rename("revenue_million_aed")

Chunked computation: 0.3 s, 12 partial results
Matches the full pandas answer: True


store_emirate
Dubai             2215.4
Abu Dhabi         1580.2
Sharjah            956.2
Ajman              508.7
Ras Al Khaimah     507.1
Fujairah           314.2
Umm Al Quwain      252.4
Name: revenue_million_aed, dtype: float64

**Means need care.**
A mean does not decompose directly: the average of chunk averages is wrong whenever chunks differ in size or composition.
Carry the **sum** and the **count** instead, and divide only at the end.

In [10]:
sums, counts, chunk_means = [], [], []
for path in files:
    for chunk in pd.read_csv(path, usecols=["product", "unit_price"], chunksize=150_000):
        laptops = chunk.loc[chunk["product"] == "Laptop", "unit_price"]
        sums.append(laptops.sum())
        counts.append(laptops.count())
        chunk_means.append(laptops.mean())

true_mean = year.loc[year["product"] == "Laptop", "unit_price"].mean()
print(f"True mean laptop price:          {true_mean:,.4f}")
print(f"Sum / count across chunks:       {sum(sums) / sum(counts):,.4f}")
print(f"Average of chunk means (wrong):  {np.mean(chunk_means):,.4f}")

True mean laptop price:          5,256.3897
Sum / count across chunks:       5,256.3897
Average of chunk means (wrong):  5,256.3975


The difference here is small because our synthetic chunks are similar, but on real data, where a December chunk can look nothing like a February chunk, averaging averages can be badly wrong.

| Decomposes across chunks | Does not decompose easily |
|---|---|
| sum, count, min, max | median and other percentiles |
| mean (as sum ÷ count) | exact number of distinct customers |
| filtering rows and writing them out | sorting the whole dataset |
| group-wise sums and counts | removing duplicates across chunks |

In [11]:
# Filtering also works chunk by chunk: write every laptop sale to one smaller file.
laptop_file = DATA_DIR / "laptop_sales_2025.csv"
first_write = True
for path in files:
    for chunk in pd.read_csv(path, chunksize=200_000):
        laptop_rows = chunk[chunk["product"] == "Laptop"]
        laptop_rows.to_csv(laptop_file, mode="w" if first_write else "a", header=first_write, index=False)
        first_write = False

print(f"Laptop sales written: {sum(1 for _ in open(laptop_file)) - 1:,} rows")

Laptop sales written: 149,647 rows


> 💭 **Quick thought question:** Chunking keeps memory low, but the loop processes one chunk after another on a single processor core.
What would you want to change to make it faster on a laptop with eight cores?

### 🧩 Guided practice: Chunked maximum

Using chunks, find the largest single transaction `total` in each `category` for the whole year.
*Hint:* collect `chunk.groupby("category")["total"].max()` from every chunk, then combine with `.groupby(level=0).max()`.
Verify your answer against `year`.

In [12]:
# TODO: Chunked maximum by category.

## 5. Lazy evaluation 💤

pandas is **eager**: `df.groupby(...).sum()` computes immediately.
Dask is **lazy**: each command only adds a step to a plan, called a **task graph**, and nothing is read or calculated until you ask for the result with `.compute()`.

Think of the difference between cooking each step of a recipe the moment you read it and reading the whole recipe first.
Reading first lets you notice that you only need two ingredients from the fridge, that two steps can happen at the same time, and that a step can be skipped.
Dask does exactly that with your code: it reads only the columns the plan needs, runs independent tasks in parallel, and avoids holding everything in memory at once.

You have already met laziness in Python: `range()` and generator expressions produce values only when asked.

In [13]:
# A generator expression is lazy: creating it is instant, even for a billion items.
start = time.perf_counter()
squares = (n * n for n in range(1_000_000_000))
print(f"Created in {time.perf_counter() - start:.6f} s ->", squares)

# Values are produced only when requested.
print("First five:", [next(squares) for _ in range(5)])

Created in 0.000034 s -> <generator object <genexpr> at 0x10f3a8450>
First five: [0, 1, 4, 9, 16]


### 🧭 Syntax first: Dask DataFrames

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
import dask.dataframe as dd

ddf = dd.read_csv("folder/transactions_*.csv")   # many files, one lazy DataFrame
ddf = dd.read_parquet("folder/file.parquet")

result = ddf[ddf["col"] == value].groupby("key")["value"].sum()   # still lazy
result.compute()                       # run the plan, return a pandas object
dask.compute(result_a, result_b)       # run several plans, sharing the work
filtered = ddf[...].persist()          # run now and KEEP the result in memory
```

Documentation: [Dask DataFrame](https://docs.dask.org/en/stable/dataframe.html) · [Best practices](https://docs.dask.org/en/stable/dataframe-best-practices.html)

In [14]:
start = time.perf_counter()
ddf = dd.read_csv(DATA_DIR / "transactions_2025-*.csv")
print(f"'Reading' 12 files took {time.perf_counter() - start:.3f} s: only the first rows were peeked at.")
print("Type:", type(ddf).__name__, "| partitions:", ddf.npartitions)
ddf

'Reading' 12 files took 0.005 s: only the first rows were peeked at.
Type: DataFrame | partitions: 12


,transaction_id,date,customer_id,store_emirate,product,category,quantity,unit_price,discount_pct,payment_method,total
npartitions=12,,,,,,,,,,,
,int64,string,int64,string,string,string,int64,float64,float64,string,float64
,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...


The display shows column names and types, but no data: Dask has only *described* the table.
Each **partition** is an ordinary pandas DataFrame that Dask will load when needed; here, one per file.

In [15]:
# head() is cheap: it computes only what it needs from the first partition.
ddf.head()

,transaction_id,date,customer_id,store_emirate,product,category,quantity,unit_price,discount_pct,payment_method,total
0,1000000,2025-01-14,4581,Abu Dhabi,Keyboard,Computing,5,284.71,5.0,Card,1423.55
1,1000001,2025-01-12,35701,Fujairah,Laptop,Computing,2,4438.68,10.0,Cash,8877.36
2,1000002,2025-01-11,50706,Ras Al Khaimah,Headphones,Accessories,4,1332.78,0.0,Card,5331.12
3,1000003,2025-01-12,63188,Dubai,Headphones,Accessories,2,727.71,0.0,Card,1455.42
4,1000004,2025-01-03,74030,Dubai,Monitor,Computing,3,2389.01,0.0,Instalments,7167.03


In [16]:
# Build a plan: computing-category revenue by emirate. Nothing runs yet.
computing_revenue = (
    ddf[ddf["category"] == "Computing"]
    .groupby("store_emirate")["total"]
    .sum()
)

print("Type:", type(computing_revenue).__name__, "| values computed so far: none")
print("Tasks in the graph:", len(computing_revenue.__dask_graph__()))

Type: Series | values computed so far: none
Tasks in the graph: 87


In [17]:
start = time.perf_counter()
result = computing_revenue.compute()       # now the files are read, in parallel
print(f"compute() took {time.perf_counter() - start:.1f} s and returned a {type(result).__name__}")

check = year[year["category"] == "Computing"].groupby("store_emirate")["total"].sum()
print("Matches pandas:", np.allclose(result.sort_index(), check.sort_index()))
(result.sort_values(ascending=False) / 1e6).round(1).rename("revenue_million_aed")

compute() took 0.3 s and returned a Series
Matches pandas: True


store_emirate
Dubai             1124.9
Abu Dhabi          799.2
Sharjah            484.2
Ajman              259.2
Ras Al Khaimah     256.5
Fujairah           159.8
Umm Al Quwain      128.2
Name: revenue_million_aed, dtype: float64

Because the plan is known in advance, Dask optimized it before running: it read only the three columns the plan uses (`category`, `store_emirate`, `total`) instead of all eleven.
This **column projection** is one of the main payoffs of laziness.

In [18]:
# Two results from the same files: compute them TOGETHER so the files are read once.
revenue_by_payment = ddf.groupby("payment_method")["total"].sum()
average_discount = ddf.groupby("category")["discount_pct"].mean()

start = time.perf_counter()
revenue_by_payment.compute()
average_discount.compute()
separately = time.perf_counter() - start

start = time.perf_counter()
payment_result, discount_result = dask.compute(revenue_by_payment, average_discount)
together = time.perf_counter() - start

print(f"Two separate compute() calls: {separately:.1f} s | one dask.compute() call: {together:.1f} s")
display(discount_result.round(2))

Two separate compute() calls: 0.6 s | one dask.compute() call: 0.5 s


category
Accessories     9.99
Computing      10.01
Mobile         10.01
Name: discount_pct, dtype: float64

In [19]:
# persist() runs the plan now and keeps the result in memory for repeated use.
laptops = ddf[ddf["product"] == "Laptop"].persist()

start = time.perf_counter()
laptop_summary = laptops.groupby("store_emirate").agg(
    sales=("total", "size"), revenue=("total", "sum"), average_price=("unit_price", "mean")
).compute()
print(f"Query on persisted data: {time.perf_counter() - start:.2f} s")
laptop_summary.sort_values("revenue", ascending=False).round(0)

Query on persisted data: 0.02 s


,sales,revenue,average_price
store_emirate,,,
Dubai,52523,826792968.0,5254.0
Abu Dhabi,37280,586722958.0,5257.0
Sharjah,22422,356378544.0,5274.0
Ajman,11999,190501883.0,5246.0
Ras Al Khaimah,11967,189280027.0,5260.0
Fujairah,7439,117671590.0,5260.0
Umm Al Quwain,6017,94271772.0,5218.0


> 💭 **Quick thought question:** A student writes a loop over the seven emirates and calls `.compute()` inside the loop for each one.
Why is that slow, and how would you rewrite it?

### 🧩 Guided practice: A lazy pipeline

Build a lazy Dask expression for the **average quantity per transaction by product** for card payments only.
Print the lazy object and its number of tasks *before* computing, then compute it and sort the result.

In [20]:
# TODO: Filter, group, average, inspect, then compute.

## 6. Distributed computing basics 🌐

Dask splits work into many small **tasks** and hands them to **workers**.
The vocabulary is the same whether the workers are the cores of your laptop or hundreds of machines in a data center.

| Term | Meaning |
|---|---|
| **Partition** | one piece of the dataset (a regular pandas DataFrame) |
| **Task** | one small job, such as "sum the `total` column of partition 3" |
| **Task graph** | all tasks plus the order they depend on each other |
| **Scheduler** | the coordinator that decides which worker runs which task, and when |
| **Worker** | a process (or thread) that executes tasks and holds partitions in memory |
| **Cluster** | a scheduler plus its workers, on one machine or many |

```text
                         ┌──> worker 1: partitions 1, 4, 7, 10 ──┐
your code ──> scheduler ─┼──> worker 2: partitions 2, 5, 8, 11 ──┼──> combine ──> result
   (plan)                └──> worker 3: partitions 3, 6, 9, 12 ──┘
```

The key idea is **move the computation to the data**: each worker summarizes the partitions it holds, and only the small summaries travel back to be combined.

In [21]:
# Each partition is an ordinary pandas DataFrame.
rows_per_partition = ddf.map_partitions(len).compute()
print("Rows per partition:", rows_per_partition.tolist())
print("Type of one partition:", type(ddf.partitions[0].compute()).__name__)

Rows per partition: [100000, 100000, 100000, 100000, 100000, 100000, 100000, 100000, 100000, 100000, 100000, 100000]
Type of one partition: DataFrame


### Schedulers on a single machine

Dask can run the same task graph in different ways.
`"synchronous"` runs one task at a time (useful for debugging), `"threads"` uses several threads that share memory (the default for DataFrames), and `"processes"` uses separate Python processes, which helps when tasks run lots of pure-Python code.

In [22]:
plan = ddf.groupby(["store_emirate", "category"])["total"].sum()

for scheduler in ["synchronous", "threads", "processes"]:
    start = time.perf_counter()
    answer = plan.compute(scheduler=scheduler)
    print(f"{scheduler:>11}: {time.perf_counter() - start:5.2f} s, {len(answer)} groups")

synchronous:  0.69 s, 21 groups


    threads:  0.34 s, 21 groups


  processes:  1.16 s, 21 groups


Exact timings depend on your computer, and the parallel schedulers have start-up and coordination costs, so on small data they may not win by much.
The point is that the **same code** can use one core, all cores, or a cluster without changes.

### A local cluster with `dask.distributed`

The `distributed` scheduler runs a real scheduler and real worker processes, even on a laptop.
It also provides a live **dashboard** in your browser that shows tasks being executed, memory per worker, and progress.
The same `Client` code connects to a cluster of many machines; only the address changes.

In [23]:
try:
    from dask.distributed import Client, LocalCluster
except ImportError:
    Client = None
    print("dask.distributed is not installed. Run: conda install -c conda-forge distributed")

if Client is not None:
    cluster = LocalCluster(n_workers=2, threads_per_worker=2, memory_limit="2GiB", dashboard_address=":0")
    client = Client(cluster)   # from now on, .compute() runs on this cluster
    print(client)
    print("Dashboard:", client.dashboard_link, "(open it in a browser while the next cell runs)")

<Client: 'tcp://127.0.0.1:50079' processes=2 threads=4, memory=4.00 GiB>
Dashboard: http://127.0.0.1:50078/status (open it in a browser while the next cell runs)


In [24]:
if Client is not None:
    start = time.perf_counter()
    monthly = (
        ddf.assign(month=ddf["date"].str.slice(0, 7))
        .groupby(["month", "category"])["total"]
        .sum()
        .compute()
    )
    print(f"Computed on {len(client.scheduler_info()['workers'])} workers in {time.perf_counter() - start:.1f} s")
    display((monthly.unstack() / 1e6).round(1).head())

Computed on 2 workers in 0.5 s


category,Accessories,Computing,Mobile
month,,,
2025-01,34.8,269.0,226.2
2025-02,35.9,268.0,223.6
2025-03,35.4,270.8,224.7
2025-04,35.0,267.3,226.3
2025-05,35.3,266.5,225.0


### Parallelizing your own functions with `dask.delayed`

Not every job is a DataFrame operation.
`dask.delayed` turns an ordinary Python function call into a lazy task, so a loop over files becomes a task graph that can run in parallel.

In [25]:
def summarize_month(path):
    # Read one monthly file and return a one-row summary.
    df = pd.read_csv(path, usecols=["date", "total", "customer_id"])
    return {
        "month": df["date"].iloc[0][:7],
        "transactions": len(df),
        "revenue_million_aed": round(df["total"].sum() / 1e6, 2),
        "distinct_customers": df["customer_id"].nunique(),
    }


start = time.perf_counter()
sequential = [summarize_month(path) for path in files]
sequential_seconds = time.perf_counter() - start

lazy_summaries = [dask.delayed(summarize_month)(path) for path in files]   # 12 lazy tasks, nothing runs yet
start = time.perf_counter()
parallel = dask.compute(*lazy_summaries)
parallel_seconds = time.perf_counter() - start

print(f"Sequential loop: {sequential_seconds:.1f} s | dask.delayed: {parallel_seconds:.1f} s")
pd.DataFrame(parallel)

Sequential loop: 0.3 s | dask.delayed: 0.1 s


,month,transactions,revenue_million_aed,distinct_customers
0,2025-01,100000,529.97,57125
1,2025-02,100000,527.53,57146
2,2025-03,100000,530.86,57223
3,2025-04,100000,528.57,57091
4,2025-05,100000,526.87,57144
5,2025-06,100000,527.32,57050
6,2025-07,100000,525.32,57136
7,2025-08,100000,527.98,57182
8,2025-09,100000,528.64,57011
9,2025-10,100000,526.04,57007


In [26]:
# Always shut the cluster down when you are finished, to free its memory and processes.
if Client is not None:
    client.close()
    cluster.close()
    print("Cluster closed.")

Cluster closed.


**From a laptop to a real cluster.**
On a university HPC system, a cloud platform, or a Kubernetes cluster, you create the cluster with a different helper (for example `dask-jobqueue` for Slurm, or a managed cloud service) and connect with `Client(address)`.
The DataFrame code above does not change.
Apache Spark (through PySpark) is the other widely used engine for distributed data processing; organizations already built on Spark usually stay with it, while Dask offers the shortest path from pandas code.

> 💭 **Quick thought question:** In the distributed setting, why is it better to send a small summary from each worker to the scheduler than to send each worker's raw partitions?

## 7. Limits and choosing the right tool 🧭

Dask is not a drop-in replacement for every pandas feature.

In [27]:
# Row positions do not exist globally across partitions, so iloc for rows is not supported.
try:
    ddf.iloc[5]
except NotImplementedError as error:
    print("NotImplementedError:", str(error)[:120])

NotImplementedError: 'DataFrame.iloc' only supports selecting columns. It must be used like 'df.iloc[:, column_indexer]'.


- **No positional row access** (`iloc` for rows), because rows are spread over partitions.
- **Sorting and exact percentiles are expensive**, because they need to see all the data; Dask's `quantile()` is approximate.
- **`.compute()` returns pandas**, so the *result* must fit in memory; aggregate before you compute.
- **Small data is slower in Dask** than in pandas, because of the overhead of planning and scheduling.

| Situation | Reasonable choice |
|---|---|
| Data fits comfortably in memory | plain pandas |
| Fits, but memory is tight or reads are slow | pandas + `category`/smaller types + Parquet |
| Too big for memory, and a simple sum or filter is all you need | chunked `read_csv` |
| Too big for memory, with joins, group-bys, and many steps | Dask DataFrame on all your cores |
| Too big for one machine, or many users | Dask on a cluster, or Spark |

## 8. Bringing it together: a monthly management report 📝

The pipeline below reads all twelve files lazily, cleans and enriches the data, aggregates it, and computes once.
Only the small final table is brought into pandas and saved.

In [28]:
report = (
    dd.read_csv(DATA_DIR / "transactions_2025-*.csv")
    .assign(discount_pct=lambda d: d["discount_pct"].fillna(0))       # cleaning: missing discount = none
    .assign(net_revenue=lambda d: d["total"] * (1 - d["discount_pct"] / 100),
            month=lambda d: d["date"].str.slice(0, 7))
    .groupby(["month", "store_emirate"])["net_revenue"]
    .sum()
    .compute()                                                         # one compute, at the end
)

monthly_table = (report.unstack("store_emirate").sort_index() / 1e6).round(2)[EMIRATES]
monthly_table.to_csv(DATA_DIR / "net_revenue_by_month_and_emirate.csv")
monthly_table.head()

store_emirate,Dubai,Abu Dhabi,Sharjah,Ajman,Ras Al Khaimah,Fujairah,Umm Al Quwain
month,,,,,,,
2025-01,170.63,120.82,73.62,39.57,38.39,23.11,19.52
2025-02,169.68,120.69,72.87,37.99,39.08,23.84,19.56
2025-03,171.32,121.38,72.57,39.26,38.29,23.56,19.91
2025-04,168.81,121.50,74.00,38.96,38.11,23.86,18.83
2025-05,168.43,118.41,74.23,38.76,39.02,24.63,19.41


## Comprehensive exercises 📝

For each exercise, show your code, the result, and one or two sentences explaining what you learned about the tool, not only about the data.

### Exercise 1: Chunked averages done right

Using chunked reading only, compute the **average discount** and the **total revenue** for each `payment_method` over the whole year.
Treat missing discounts as missing (do not fill them).
Verify both results against the full `year` DataFrame, and explain why you had to carry sums and counts.

In [29]:
# Write your solution here.

### Exercise 2: Why medians are hard

Compute the median `unit_price` of each monthly file, then take the median of those twelve medians.
Compare it with the true median from `year` and with Dask's approximate `ddf["unit_price"].quantile(0.5).compute()`.
Which methods agree, and why can a "median of medians" go wrong?

In [30]:
# Write your solution here.

### Exercise 3: Inspect, optimize, compute

Build two lazy results from `ddf`: (a) the number of transactions per product in Dubai, and (b) the net revenue per category in Dubai.
Print each lazy object and its number of tasks.
Compute them once separately and once together with `dask.compute()`, time both approaches, and explain the difference.

In [31]:
# Write your solution here.

### Exercise 4: Parallel processing of files

Write a function `top_product(path)` that returns the month and the product with the highest revenue in that monthly file.
Run it over all twelve files first with a regular loop and then with `dask.delayed`, compare the run times, and show the results in one DataFrame.

In [32]:
# Write your solution here.

### Exercise 5: A tool-choice memo

For each scenario, recommend pandas, pandas with chunking, Dask on a laptop, or a cluster (Dask or Spark), and justify your choice in two sentences:

1. A 40 MB customer survey that a marketing team analyzes weekly.
2. A 25 GB log of website visits from which you need the daily count of visits.
3. Five years of transactions (60 GB in monthly Parquet files) that must be joined to a customer table and summarized by customer segment.
4. A 5 TB stream of sensor readings that several analysts query every day.

In [33]:
# Write your answers as comments or in a Markdown cell.

## Summary ✅

- pandas is eager and in-memory; when data approaches the size of your RAM, first **shrink** it with better types and Parquet.
- **Chunking** processes one piece at a time; it works for calculations that decompose (sums, counts, min/max, means as sum ÷ count, filters) and needs care for medians, distinct counts, sorting, and deduplication.
- **Lazy evaluation** builds a task graph first and runs it only on `.compute()`, which lets Dask read only the needed columns, run tasks in parallel, and share work with `dask.compute()`; `.persist()` keeps an intermediate result in memory.
- **Distributed computing** splits data into partitions and work into tasks that a scheduler sends to workers; the same Dask code runs on threads, processes, a local cluster, or many machines.
- Choose the simplest tool that works: pandas for data that fits, Dask (or Spark) when it does not.